# Trắc nghiệm pháp luật — TF-IDF (BẢN TỐT NHẤT ~57.91%) | Colab

> **Tuân thủ: KHÔNG dùng deep learning, KHÔNG dùng pretrained model.** Chỉ TF-IDF (scikit-learn).
> Đây là phiên bản đạt điểm cao nhất. Toàn bộ code trong **một notebook này**. Chạy **CPU ~1-2 phút**.

**Cách chạy:** Runtime → **Run all**. Khi cell *Upload* hiện, tải lên 2 file: **dataset (corpus)** và **de_thi** (tên tùy ý).

## 1. Thư viện (có sẵn trên Colab)

In [ ]:
import sklearn, numpy
print("scikit-learn", sklearn.__version__, "| numpy", numpy.__version__)

## 2. Tải dữ liệu (corpus + đề thi)

In [ ]:
from google.colab import files
print(">> Chọn 2 file: dataset(corpus).json  và  de_thi.json")
up = files.upload()
print("Đã tải:", list(up.keys()))

## 3. Toàn bộ mã giải pháp

In [ ]:
# =============================================================
#  TRẮC NGHIỆM PHÁP LUẬT — TF-IDF (truy xuất gộp) | KHÔNG DL, KHÔNG pretrained
#  >>> Đây là phiên bản đạt điểm cao nhất (~57.91%). Nhanh ~1-2 phút CPU. <<<
#
#  Mỗi câu:
#   - Truy xuất 1 lần: query = "câu hỏi×2 + 4 phương án" -> top-90 điều luật (pool).
#   - Chấm mỗi phương án trên pool:
#       rel(d) = max( cos_tfidf(câu hỏi, d) , 0.3·cos_tfidf(câu hỏi+đáp án×2, d) )
#       điểm   = best_mean_d [ rel(d)·coverage(opt,d) + 0.1·rel(d)·substring(opt,d) ]
#   - Câu phủ định ("không đúng","ngoại trừ","sai"...) -> chọn phương án ÍT được ủng hộ nhất.
# =============================================================
import re, json, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

CONTENT_TRUNC = 1500
TOP_K = 90
W_REL_OPT = 0.3
W_SUB = 0.10
MAXF = 300_000
CHOICES = ["A", "B", "C", "D"]
NEG_PATTERNS = ["không đúng", "không chính xác", "không phải", "không thuộc", "không bao gồm",
    "không nằm trong", "không được phép", "không phù hợp", "ngoại trừ", "loại trừ",
    "nào sai", "phát biểu sai", "ý nào sai", "điều nào sai", "đáp án nào sai",
    "phương án nào sai", "khẳng định sai", "nhận định sai", "sai là"]

def normalize(t): return re.sub(r"\s+", " ", str(t or "").lower().strip())
def tokenize(t):  return re.findall(r"\w+", t, flags=re.UNICODE)
def is_negation(q): return any(p in q for p in NEG_PATTERNS)
def best_mean(v):
    if v.size == 0: return 0.0
    t3 = np.partition(v, -3)[-3:] if v.size > 3 else v
    return 0.85 * float(t3.max()) + 0.15 * float(t3.mean())

def load_corpus(corpus_file):
    """Đọc corpus (JSONL/JSON array). Ghép title×2 + demuc + chude + content(cắt)."""
    docs = []
    with open(corpus_file, "r", encoding="utf-8") as f:
        first = f.read(1); f.seek(0)
        rows = json.load(f) if first == "[" else f
        for d in rows:
            if isinstance(d, str):
                d = d.strip()
                if not d: continue
                try: d = json.loads(d)
                except Exception: continue
            ti = normalize(d.get("title", "")); de = normalize(d.get("demuc_name", ""))
            ch = normalize(d.get("chude_name", "")); co = normalize(d.get("content", ""))[:CONTENT_TRUNC]
            txt = f"{ti} {ti} {de} {ch} {co}".strip()
            if txt: docs.append(txt)
    return docs

def answer_question(item, documents, vectorizer, doc_vectors, cache):
    qn = normalize(item.get("question", "")); ch = {c: normalize(item.get(c, "")) for c in CHOICES}
    rt = " ".join([qn, qn] + [ch[c] for c in CHOICES]).strip()
    sims = (doc_vectors @ vectorizer.transform([rt]).T).toarray().ravel()
    k = min(TOP_K, sims.shape[0]); part = np.argpartition(sims, -k)[-k:]
    pool = part[np.argsort(sims[part])][::-1]
    pv = doc_vectors[pool]
    qcos = (pv @ vectorizer.transform([qn]).T).toarray().ravel()
    hyp = [f"{qn} {ch[c]} {ch[c]}".strip() for c in CHOICES]
    hc = (pv @ vectorizer.transform(hyp).T).toarray().T          # (4, P)
    for di in pool:
        di = int(di)
        if di not in cache: cache[di] = set(tokenize(documents[di]))
    sc = []
    for ci, c in enumerate(CHOICES):
        ct = ch[c]
        if not ct: sc.append(-1e9); continue
        cw = set(tokenize(ct)); vals = np.empty(len(pool), np.float32)
        for j, di in enumerate(pool):
            di = int(di); rel = max(qcos[j], W_REL_OPT * hc[ci, j])
            cov = len(cw & cache[di]) / len(cw) if cw else 0.0
            sub = 1.0 if ct in documents[di] else 0.0
            vals[j] = rel * cov + W_SUB * rel * sub
        sc.append(best_mean(vals))
    return CHOICES[int(np.argmin(sc)) if is_negation(qn) else int(np.argmax(sc))]

def make_submission(test_file, corpus_file, output_file="submission.json"):
    documents = load_corpus(corpus_file)
    print(f"Đã tải {len(documents)} điều luật. Đang xây dựng TF-IDF...")
    vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=MAXF,
        sublinear_tf=True, token_pattern=r"(?u)\w+", lowercase=True, dtype=np.float32)
    doc_vectors = vectorizer.fit_transform(documents)            # L2-normalized -> dot = cosine
    print(f"TF-IDF: {doc_vectors.shape[0]} x {doc_vectors.shape[1]} — sẵn sàng.")
    with open(test_file, "r", encoding="utf-8") as f: test_data = json.load(f)
    cache = {}; submissions = []
    for item in tqdm(test_data):
        submissions.append({"id": int(item.get("id")),
                            "answer": answer_question(item, documents, vectorizer, doc_vectors, cache)})
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(submissions, f, ensure_ascii=False, indent=2)
    print(f"Đã ghi {len(submissions)} đáp án -> {output_file}")
    return submissions

## 4. Chạy sinh `submission.json`

In [ ]:
import glob, os, time
corpus = next((f for f in glob.glob('*.json') if 'dataset' in f.lower() or os.path.getsize(f) > 5_000_000), None)
test   = next((f for f in glob.glob('*.json') if 'de_thi' in f.lower() or 'dethi' in f.lower()), None)
print("corpus =", corpus, "| test =", test)
assert corpus and test, "Không tìm thấy file! Hãy upload đúng 2 file .json (corpus + de_thi)."
t0 = time.time()
make_submission(test_file=test, corpus_file=corpus, output_file='submission.json')
print("Thời gian: %.0f giây" % (time.time() - t0))

## 5. Kiểm tra & đóng gói `submission.zip`

In [ ]:
import json, glob, zipfile
from collections import Counter
sub = json.load(open('submission.json', encoding='utf-8'))
tst = json.load(open(test, encoding='utf-8'))
assert len(sub) == len(tst), f"Thiếu đáp án: {len(sub)}/{len(tst)}"
assert all(s['answer'] in ['A', 'B', 'C', 'D'] for s in sub), "Có đáp án không hợp lệ"
print(f"OK: {len(sub)} đáp án | id khớp đề:", {s['id'] for s in sub} == {int(t['id']) for t in tst})
print("Phân bố:", dict(sorted(Counter(s['answer'] for s in sub).items())))
nb = next(iter(glob.glob('*.ipynb')), None)
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('submission.json')
    if nb: z.write(nb)
print("Đã tạo submission.zip:", zipfile.ZipFile('submission.zip').namelist())
from google.colab import files
files.download('submission.zip')